In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from datetime import datetime, timezone
import os
from load_dotenv import load_dotenv
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import lit


load_dotenv()

aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "us-east-1"
# os.environ.setdefault("AWS_REGION", aws_region)
# os.environ.setdefault("AWS_DEFAULT_REGION", aws_region)

# # GlueCatalog uses the AWS SDK credential chain, so expose the credentials
# # loaded from .env using the standard AWS environment variable names.
# if os.getenv("aws_access_key_id") and not os.getenv("AWS_ACCESS_KEY_ID"):
#     os.environ["AWS_ACCESS_KEY_ID"] = os.environ["aws_access_key_id"]
# if os.getenv("aws_secret_access_key") and not os.getenv("AWS_SECRET_ACCESS_KEY"):
#     os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ["aws_secret_access_key"]


In [2]:
catalog_name = "glue_catalog"
database_name = "bronze"
bucket_name = "amzn-s3-job-prj"
warehouse_path = f"s3://{bucket_name}/"

spark = SparkSession.builder \
    .appName("iceberg-s3-aws") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    ) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config(f"spark.sql.catalog.{catalog_name}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{catalog_name}.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config(f"spark.sql.catalog.{catalog_name}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config(f"spark.sql.catalog.{catalog_name}.warehouse", warehouse_path) \
    .config(f"spark.sql.catalog.{catalog_name}.client.region", aws_region) \
    .config("spark.sql.defaultCatalog", catalog_name) \
    .config("spark.hadoop.fs.s3a.region", aws_region) \
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{aws_region}.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider") \
    .config("spark.driver.extraJavaOptions", f"-Daws.region={aws_region}") \
    .config("spark.executor.extraJavaOptions", f"-Daws.region={aws_region}") \
    .getOrCreate()


your 131072x1 screen size is bogus. expect trouble
26/05/15 17:46:29 WARN Utils: Your hostname, kien resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/15 17:46:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/k/job_ete/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ntk04/.ivy2/cache
The jars for the packages stored in: /home/ntk04/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5151d834-5308-4782-bcf8-46ddc5029d96;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.5.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 418ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.5.0 from centra

In [3]:
query = f"CREATE DATABASE IF NOT EXISTS {catalog_name}.{database_name}"


In [4]:
spark.sql(query)


DataFrame[]

In [5]:
now = datetime.now(timezone.utc)
today = now.strftime("%Y-%m-%d")

In [6]:
# Đọc JSON
df = spark.read.json(f"s3a://amzn-s3-job-prj/raw/topcv_jobs/dt={today}/*.jsonl")  

26/05/15 17:46:51 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
df.show()

+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+
|             company|          crawl_time|   experience|     job_description| job_id|             job_url|            location|       salary|              skills|source_page|               title|
+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+
|CÔNG TY TNHH VIP ...|2026-05-15T17:45:...|        1 năm|Chịu trách nhiệm ...|2157201|https://www.topcv...|- Hà Nội: Tòa nhà...|             |Tốt nghiệp ĐH chí...|          1|Nhân Viên Kinh Do...|
|Công ty Cổ phần G...|2026-05-15T17:45:...|Không yêu cầu|Tham gia phân tíc...|2159859|https://www.topcv...|- Vĩnh Long: Phườ...|   Thoả thuận|Yêu cầu kỹ thuật ...|          1|Lập Trình Viên TR...|
|CÔNG TY TNHH C

In [8]:
def partition_by_crawl_time(s):
    s = s.split("T")[0]  # Lấy phần ngày
    return s

partition_by_crawl_time_udf = udf(partition_by_crawl_time, StringType())

In [9]:
df = df.withColumn("ingestion_time", current_timestamp()) \
    .withColumn("dt", partition_by_crawl_time_udf(col("crawl_time"))) \
    

In [10]:
df.show()

+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+
|             company|          crawl_time|   experience|     job_description| job_id|             job_url|            location|       salary|              skills|source_page|               title|      ingestion_time|        dt|
+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+
|CÔNG TY TNHH VIP ...|2026-05-15T17:45:...|        1 năm|Chịu trách nhiệm ...|2157201|https://www.topcv...|- Hà Nội: Tòa nhà...|             |Tốt nghiệp ĐH chí...|          1|Nhân Viên Kinh Do...|2026-05-15 17:47:...|2026-05-15|
|Công ty Cổ phần G...|2026-05-15T17:45:...|Không yêu cầu|Tham gia phân tíc...|215985

In [11]:
df.writeTo(f"{catalog_name}.{database_name}.jobs") \
    .using("iceberg") \
    .partitionedBy("dt") \
    .createOrReplace()


In [12]:
# Append later with:
# df.writeTo(f"{catalog_name}.{database_name}.jobs").append()


In [13]:
df = spark.table(f"{catalog_name}.{database_name}.jobs")

df.show()


+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+
|             company|          crawl_time|   experience|     job_description| job_id|             job_url|            location|       salary|              skills|source_page|               title|      ingestion_time|        dt|
+--------------------+--------------------+-------------+--------------------+-------+--------------------+--------------------+-------------+--------------------+-----------+--------------------+--------------------+----------+
|CÔNG TY TNHH VIP ...|2026-05-15T17:45:...|        1 năm|Chịu trách nhiệm ...|2157201|https://www.topcv...|- Hà Nội: Tòa nhà...|             |Tốt nghiệp ĐH chí...|          1|Nhân Viên Kinh Do...|2026-05-15 17:47:...|2026-05-15|
|Công ty Cổ phần G...|2026-05-15T17:45:...|Không yêu cầu|Tham gia phân tíc...|215985